In [ ]:
"""
build_clean_dataset_v5.py
==========================
Two fixes applied:

FIX 1 — Normalize k-mer vectors to frequencies
  Divide each read's k-mer counts by its total k-mer count.
  This removes the effect of read length differences.
  The vector now represents RELATIVE k-mer composition,
  not absolute counts.

FIX 2 — Diverse healthy training data
  Adds 1000 Genomes samples (NA12878, NA19238) to the
  healthy training class alongside human.fastq.
  The model must learn that all of these are healthy,
  not just the specific k-mer profile of one file.

Output: ml_ready_v5/
"""

from google.colab import drive
drive.mount("/content/drive")

import os, random, gzip
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/DNA_database_For_training"
HUMAN_FASTQ = f"{BASE_DIR}/human.fastq"
SARS_FASTQ  = f"{BASE_DIR}/sars_training.fastq"
OUT_DIR     = f"{BASE_DIR}/ml_ready_v5"

# Additional healthy human sources from 1000 Genomes
EXTRA_HEALTHY = [
    f"{BASE_DIR}/validation_human/NA12878.fastq.gz",
    f"{BASE_DIR}/validation_human/NA19238.fastq.gz",
]

K                = 4
READS_PER_SAMPLE = 50
TRAIN_FRACTION   = 0.70
SEED             = 42
INFECTED_RATIOS  = [0.10, 0.30, 0.50, 0.60]
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
rng     = random.Random(SEED)
VEC_LEN = 4 ** K

BASE_BITS = {"A": 0, "C": 1, "G": 2, "T": 3}
VALID     = set(BASE_BITS)

log_lines = []
def log(msg=""):
    print(msg)
    log_lines.append(msg)


# ── Cleaning ──────────────────────────────────────────────────────────────────
def clean_seq(seq: str) -> str:
    return "".join(b for b in seq.upper() if b in VALID)


# ── FASTQ loader ──────────────────────────────────────────────────────────────
def load_and_clean(path: str, label: str) -> list:
    seqs    = []
    open_fn = gzip.open if path.endswith(".gz") else open
    with open_fn(path, "rt", errors="replace") as f:
        lines = []
        for line in f:
            lines.append(line.rstrip("\n").rstrip("\r"))
            if len(lines) == 4:
                _, seq, _, _ = lines
                lines = []
                cleaned = clean_seq(seq)
                if len(cleaned) >= K:
                    seqs.append(cleaned)
    log(f"  {label}: {len(seqs):,} reads loaded and cleaned")
    return seqs

## ACCTTGGAAA -   k=1 ACCT CCTT CTTG
#Converts a 4-letter k-mer like ACGT to an integer 0–255 by packing 2 bits per base
 #(A=00, C=01, G=10, T=11). So AAAA→0, AAAC→1, … TTTT→255.
 #This is just a fast hash from k-mer string to vector index.
# ── k-mer encoding ────────────────────────────────────────────────────────────
def kmer_to_index(kmer):
    idx = 0
    for b in kmer:
        idx = (idx << 2) | BASE_BITS[b]
    return idx

def encode_sequence(seq: str) -> np.ndarray:
    vec = np.zeros(VEC_LEN, dtype=np.float32)
    for i in range(len(seq) - K + 1):
        vec[kmer_to_index(seq[i: i + K])] += 1
    return vec

def normalize_vec(vec: np.ndarray) -> np.ndarray:
    """
    FIX 1: Convert k-mer counts to frequencies.
    Divide by total count so the vector sums to 1.
    This removes read-length effects — a short read and a long read
    with the same composition will have the same normalized vector.
    """
    total = vec.sum()
    if total > 0:
        return vec / total
    return vec

def reads_to_sample(reads: list) -> np.ndarray:
    """
    Encode, normalize, then average.
    Normalization happens per-read BEFORE averaging so each read
    contributes equally regardless of its length.
    """
    vecs = np.stack([normalize_vec(encode_sequence(r)) for r in reads], axis=0)
    return vecs.mean(axis=0)


# ── Sample builders ───────────────────────────────────────────────────────────
def build_healthy_samples(human_seqs, reads_per_sample, rng):
    shuffled = human_seqs[:]
    rng.shuffle(shuffled)
    n = len(shuffled) // reads_per_sample
    return [reads_to_sample(shuffled[i*reads_per_sample:(i+1)*reads_per_sample])
            for i in range(n)]

def build_infected_samples(human_seqs, sars_seqs, sars_fraction,
                           reads_per_sample, rng):
    n_sars  = max(1, int(reads_per_sample * sars_fraction))
    n_human = reads_per_sample - n_sars
    n       = min(len(human_seqs) // n_human, len(sars_seqs) // n_sars)

    h = human_seqs[:]
    s = sars_seqs[:]
    rng.shuffle(h)
    rng.shuffle(s)

    samples = []
    for i in range(n):
        chunk = h[i*n_human:(i+1)*n_human] + s[i*n_sars:(i+1)*n_sars]
        rng.shuffle(chunk)
        samples.append(reads_to_sample(chunk))
    return samples


# ═════════════════════════════════════════════════════════════════════════════
log("=" * 60)
log("  Fixed pipeline v5")
log("  Fix 1: k-mer frequency normalization (removes read length effects)")
log("  Fix 2: diverse healthy training data (1000 Genomes added)")
log("=" * 60)
log(f"  k={K}  reads_per_sample={READS_PER_SAMPLE}  output=ml_ready_v5")
log()

# ── Step 1: Load all sources ──────────────────────────────────────────────────
log("=" * 60)
log("  Step 1 — Load and clean all sources")
log("=" * 60)

human_seqs = load_and_clean(HUMAN_FASTQ, "human.fastq")
sars_seqs  = load_and_clean(SARS_FASTQ,  "sars.fastq")

# Load extra healthy sources (1000 Genomes)
extra_seqs = []
for path in EXTRA_HEALTHY:
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        seqs = load_and_clean(path, os.path.basename(path))
        extra_seqs.extend(seqs)
    else:
        log(f"  SKIP: {os.path.basename(path)} not found or empty")

# Combine all healthy human sources
all_healthy_seqs = human_seqs + extra_seqs
log(f"\n  Total healthy pool : {len(all_healthy_seqs):,} reads")
log(f"    human.fastq      : {len(human_seqs):,}")
log(f"    1000 Genomes     : {len(extra_seqs):,}")
log()

# ── Step 2: Build healthy samples from all human sources ─────────────────────
log("=" * 60)
log("  Step 2 — Healthy samples (class 0)")
log("=" * 60)
healthy_samples = build_healthy_samples(all_healthy_seqs, READS_PER_SAMPLE, rng)
log(f"  Healthy samples : {len(healthy_samples):,}")
log()

# ── Step 3: Build infected samples ───────────────────────────────────────────
log("=" * 60)
log("  Step 3 — Infected samples (class 1)")
log("=" * 60)
infected_samples = []
for ratio in INFECTED_RATIOS:
    batch = build_infected_samples(
        human_seqs, sars_seqs, ratio, READS_PER_SAMPLE, rng
    )
    infected_samples.extend(batch)
    n_sars_reads = max(1, int(READS_PER_SAMPLE * ratio))
    log(f"  SARS {ratio:.0%} ({n_sars_reads}/{READS_PER_SAMPLE} reads/sample) : "
        f"{len(batch):,} samples")

log(f"\n  Total infected : {len(infected_samples):,}")
log()

# ── Step 4: Combine and split ─────────────────────────────────────────────────
log("=" * 60)
log("  Step 4 — Combine, shuffle, split 70/30")
log("=" * 60)

X_all = np.concatenate([
    np.stack(healthy_samples,  axis=0).astype(np.float32),
    np.stack(infected_samples, axis=0).astype(np.float32),
], axis=0)

y_all = np.concatenate([
    np.zeros(len(healthy_samples),  dtype=np.int64),
    np.ones(len(infected_samples),  dtype=np.int64),
], axis=0)

log(f"  Total  : {len(y_all):,}")
log(f"  Class 0 (healthy)  : {(y_all==0).sum():,}  ({(y_all==0).mean()*100:.1f}%)")
log(f"  Class 1 (infected) : {(y_all==1).sum():,}  ({(y_all==1).mean()*100:.1f}%)")

idx = list(range(len(y_all)))
rng.shuffle(idx)
X_all, y_all = X_all[idx], y_all[idx]

n_train      = int(len(y_all) * TRAIN_FRACTION)
X_train, y_train = X_all[:n_train], y_all[:n_train]
X_test,  y_test  = X_all[n_train:], y_all[n_train:]

log(f"\n  Train : {len(y_train):,}  "
    f"(healthy={(y_train==0).sum():,}, infected={(y_train==1).sum():,})")
log(f"  Test  : {len(y_test):,}  "
    f"(healthy={(y_test==0).sum():,}, infected={(y_test==1).sum():,})")

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0]  == y_test.shape[0]
assert X_train.shape[1] == VEC_LEN
assert not np.isnan(X_train).any()
log("\n  Sanity checks passed.")

# ── Step 5: Save ──────────────────────────────────────────────────────────────
log()
log("=" * 60)
log("  Step 5 — Saving to Drive")
log("=" * 60)

np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUT_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(OUT_DIR, "y_test.npy"),  y_test)

with open(os.path.join(OUT_DIR, "label_map.txt"), "w") as f:
    f.write(f"k                = {K}\n")
    f.write(f"reads_per_sample = {READS_PER_SAMPLE}\n")
    f.write(f"SARS fractions   = {INFECTED_RATIOS}\n")
    f.write(f"normalization    = L1 frequency (per read)\n")
    f.write(f"healthy sources  = human.fastq + 1000 Genomes\n")
    f.write(f"0 = healthy\n1 = infected\n")

with open(os.path.join(OUT_DIR, "pipeline_log.txt"), "w") as f:
    f.write("\n".join(log_lines))

log(f"  X_train.npy  {X_train.shape}")
log(f"  y_train.npy  {y_train.shape}")
log(f"  X_test.npy   {X_test.shape}")
log(f"  y_test.npy   {y_test.shape}")
log(f"\n  Saved to: {OUT_DIR}")
log()
log("  Next steps:")
log("  1. Update train_model_final_v3.py:")
log(f"       X_PATH  = '{OUT_DIR}/X_train.npy'")
log(f"       Y_PATH  = '{OUT_DIR}/y_train.npy'")
log(f"       OUT_DIR = '{BASE_DIR}/model_v5'")
log("  2. Retrain")
log("  3. Update MODEL_PATH in encode_and_test_validation_v2.py:")
log(f"       MODEL_PATH = '{BASE_DIR}/model_v5/model_fp32.pt'")
log("  IMPORTANT: Also update encode_and_test_validation_v2.py")
log("  to normalize vectors the same way:")
log("  Replace encode_sequence() to divide by sum after counting.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Fixed pipeline v5
  Fix 1: k-mer frequency normalization (removes read length effects)
  Fix 2: diverse healthy training data (1000 Genomes added)
  k=4  reads_per_sample=50  output=ml_ready_v5

  Step 1 — Load and clean all sources
  human.fastq: 406,890 reads loaded and cleaned
  sars.fastq: 799,916 reads loaded and cleaned
  NA12878.fastq.gz: 50,000 reads loaded and cleaned
  NA19238.fastq.gz: 50,000 reads loaded and cleaned

  Total healthy pool : 506,890 reads
    human.fastq      : 406,890
    1000 Genomes     : 100,000

  Step 2 — Healthy samples (class 0)
  Healthy samples : 10,137

  Step 3 — Infected samples (class 1)
  SARS 10% (5/50 reads/sample) : 9,042 samples
  SARS 30% (15/50 reads/sample) : 11,625 samples
  SARS 50% (25/50 reads/sample) : 16,275 samples
  SARS 60% (30/50 reads/sample) : 20,344 samples

  Total infected : 57,286

  Step 4 — 

In [ ]:
"""
train_and_export_coe.py
========================
Trains a single-layer MAC array model on the fixed k-mer dataset,
then exports weights as:
  - int8 quantized .coe file for Xilinx FPGA BRAM
  - .npy backup

Architecture: one linear weight matrix (256 → 2)
Simple enough to implement as a MAC array on FPGA.

Output:
  model_coe/
    multiplexdiagnosis_weights_64bit.coe   FPGA BRAM init file
    weights_int8.npy                       backup
    weights_fp32.npy                       original float weights
    training_log.txt
"""

from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR  = "/content/drive/MyDrive/DNA_database_For_training"
X_PATH    = f"{BASE_DIR}/ml_ready_v5/X_train.npy"
Y_PATH    = f"{BASE_DIR}/ml_ready_v5/y_train.npy"
OUT_DIR   = f"{BASE_DIR}/model_coe"

EPOCHS        = 100
BATCH_SIZE    = 256
LEARNING_RATE = 0.01
VAL_FRACTION  = 0.10
SEED          = 42
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}\n")

log_lines = []
def log(msg=""):
    print(msg)
    log_lines.append(msg)


# ── Load data ─────────────────────────────────────────────────────────────────
log("Loading data ...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

n_healthy  = int((y == 0).sum())
n_infected = int((y == 1).sum())

log(f"  X shape      : {X.shape}")
log(f"  Healthy  (0) : {n_healthy:,}  ({n_healthy/len(y)*100:.1f}%)")
log(f"  Infected (1) : {n_infected:,}  ({n_infected/len(y)*100:.1f}%)")
log()


# ── DataLoaders ───────────────────────────────────────────────────────────────
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

dataset    = TensorDataset(X_t, y_t)
val_size   = int(len(dataset) * VAL_FRACTION)
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False)

log(f"Train : {train_size:,}  |  Val : {val_size:,}\n")


# ── Model ─────────────────────────────────────────────────────────────────────
class MACArrayModel(nn.Module):
    """
    Single weight matrix: input_dim → num_classes.
    Maps directly to a MAC array on FPGA.
    One matrix multiply + bias → logits → argmax = prediction.

    self.weights is the parameter extracted for FPGA deployment.
    """
    def __init__(self, input_dim=256, num_classes=2):
        super().__init__()
        self.weights = nn.Parameter(
            torch.randn(num_classes, input_dim) * 0.01
        )
        self.bias = nn.Parameter(torch.zeros(num_classes))

    def forward(self, x):
        return x @ self.weights.T + self.bias


input_dim = X.shape[1]   # 256
model     = MACArrayModel(input_dim=input_dim, num_classes=2).to(device)

n_params = sum(p.numel() for p in model.parameters())
log(f"Model  : MACArrayModel  ({input_dim} → 2)")
log(f"Params : {n_params:,}  ({n_params*4/1024:.2f} KB FP32)\n")


# ── Loss and optimiser ────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# ── Training loop ─────────────────────────────────────────────────────────────
log("=" * 65)
log("Training MAC Array Model ...")
log("=" * 65)

best_val_acc  = 0.0
best_epoch    = 0

for epoch in range(EPOCHS):

    model.train()
    tr_loss = tr_correct = tr_total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        tr_loss    += loss.item() * len(yb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total   += len(yb)

    model.eval()
    vl_correct = vl_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds       = model(xb).argmax(1)
            vl_correct += (preds == yb).sum().item()
            vl_total   += len(yb)

    tr_acc = tr_correct / tr_total * 100
    vl_acc = vl_correct / vl_total * 100

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_epoch   = epoch
        torch.save(model.state_dict(), f"{OUT_DIR}/best_model.pt")

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        log(f"Epoch {epoch:03d} | Loss: {tr_loss/tr_total:.4f} | "
            f"Train: {tr_acc:.2f}% | Val: {vl_acc:.2f}%")

log()
log(f"Training complete.")
log(f"Best val accuracy : {best_val_acc:.2f}%  (epoch {best_epoch})")
log()


# ── Reload best weights ───────────────────────────────────────────────────────
model.load_state_dict(torch.load(f"{OUT_DIR}/best_model.pt",
                                  map_location="cpu"))
model = model.cpu()
model.eval()


# ── Confusion matrix ──────────────────────────────────────────────────────────
log("=" * 65)
log("Validation confusion matrix")
log("=" * 65)

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds = model(xb).argmax(1).numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

TP = int(((all_preds==1)&(all_labels==1)).sum())
TN = int(((all_preds==0)&(all_labels==0)).sum())
FP = int(((all_preds==1)&(all_labels==0)).sum())
FN = int(((all_preds==0)&(all_labels==1)).sum())

precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0

log(f"                  Healthy    Infected")
log(f"  Actual Healthy    {TN:<10} {FP:<10} <- FP")
log(f"  Actual Infected   {FN:<10} {TP:<10} <- FN (critical)")
log(f"  Precision   : {precision*100:.2f}%")
log(f"  Recall      : {recall*100:.2f}%")
log(f"  F1          : {f1:.4f}")
log()


# ── Extract and export weights ────────────────────────────────────────────────
log("=" * 65)
log("Extracting weights for FPGA BRAM export")
log("=" * 65)

# Extract the custom weights parameter (shape: 2 x 256)
layer_weights = model.weights.detach().cpu().numpy()
layer_bias    = model.bias.detach().cpu().numpy()

log(f"  Weight shape : {layer_weights.shape}")
log(f"  Bias shape   : {layer_bias.shape}")
log(f"  Weight range : [{layer_weights.min():.4f}, {layer_weights.max():.4f}]")

# Save FP32 backup
np.save(os.path.join(OUT_DIR, "weights_fp32.npy"), layer_weights)
np.save(os.path.join(OUT_DIR, "bias_fp32.npy"),    layer_bias)
log("  Saved weights_fp32.npy")

# ── Int8 quantization ─────────────────────────────────────────────────────────
# Scale weights to 0–255 range
w_min = layer_weights.min()
w_max = layer_weights.max()

layer_weights_quant = np.round(
    255 * (layer_weights - w_min) / (w_max - w_min)
).astype(int)

log(f"\n  Int8 quantization:")
log(f"    w_min = {w_min:.6f}  w_max = {w_max:.6f}")
log(f"    scale = (w - {w_min:.6f}) / {w_max-w_min:.6f} * 255")
log(f"    quantized range : [{layer_weights_quant.min()}, {layer_weights_quant.max()}]")

np.save(os.path.join(OUT_DIR, "weights_int8.npy"), layer_weights_quant)
log("  Saved weights_int8.npy")

# Save quantization params so FPGA can dequantize if needed
quant_params_path = os.path.join(OUT_DIR, "quant_params.txt")
with open(quant_params_path, "w") as f:
    f.write(f"w_min  = {w_min}\n")
    f.write(f"w_max  = {w_max}\n")
    f.write(f"scale  = (w - w_min) / (w_max - w_min) * 255\n")
    f.write(f"dequant = (int8_val / 255) * (w_max - w_min) + w_min\n")
log("  Saved quant_params.txt")

# ── Pack into 64-bit hex words and write .coe ─────────────────────────────────
log()
log("  Packing 8 x int8 weights into 64-bit words ...")

flat_weights = layer_weights_quant.flatten()
hex_payloads = []

for i in range(0, len(flat_weights), 8):
    chunk   = flat_weights[i: i + 8]
    word_64 = 0
    for token in chunk:
        word_64 = (word_64 << 8) | int(token)
    hex_payloads.append(f"{word_64:016X}")

coe_path = os.path.join(OUT_DIR, "multiplexdiagnosis_weights_64bit.coe")
log(f"  Writing {len(hex_payloads)} 64-bit words to .coe ...")

with open(coe_path, "w") as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    for i, hex_val in enumerate(hex_payloads):
        if i == len(hex_payloads) - 1:
            f.write(f"{hex_val};\n")
        else:
            f.write(f"{hex_val},\n")

log(f"  Saved multiplexdiagnosis_weights_64bit.coe")
log(f"  Total 64-bit words : {len(hex_payloads)}")
log(f"  BRAM size required : {len(hex_payloads)*8} bytes  "
    f"({len(hex_payloads)*8/1024:.2f} KB)")
log()

# ── FPGA inference note ───────────────────────────────────────────────────────
log("=" * 65)
log("  FPGA inference flow")
log("=" * 65)
log(f"  1. Encode patient reads to k-mer frequency vectors (256 floats)")
log(f"  2. Average all read vectors → one 256-element input vector")
log(f"  3. Load weight matrix from BRAM ({layer_weights.shape[0]} x {layer_weights.shape[1]})")
log(f"  4. MAC: output = input_vector @ weights.T + bias")
log(f"  5. Argmax of 2-element output → 0=healthy  1=infected")
log()
log(f"  Dequantize weights before MAC:")
log(f"    w = (bram_int8 / 255) * ({w_max:.6f} - ({w_min:.6f})) + ({w_min:.6f})")

# ── Save log ──────────────────────────────────────────────────────────────────
with open(os.path.join(OUT_DIR, "training_log.txt"), "w") as f:
    f.write("\n".join(log_lines))

log()
log(f"All files saved to: {OUT_DIR}")
log(f"  multiplexdiagnosis_weights_64bit.coe  ← load into Vivado BRAM")
log(f"  weights_fp32.npy                      ← FP32 backup")
log(f"  weights_int8.npy                      ← int8 backup")
log(f"  quant_params.txt                      ← dequantization params")
log(f"  training_log.txt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device : cpu

Loading data ...
  X shape      : (47196, 256)
  Healthy  (0) : 7,035  (14.9%)
  Infected (1) : 40,161  (85.1%)

Train : 42,477  |  Val : 4,719

Model  : MACArrayModel  (256 → 2)
Params : 514  (2.01 KB FP32)

Training MAC Array Model ...
Epoch 000 | Loss: 0.4437 | Train: 84.64% | Val: 85.29%
Epoch 020 | Loss: 0.2286 | Train: 88.78% | Val: 88.90%
Epoch 040 | Loss: 0.1637 | Train: 93.99% | Val: 93.92%
Epoch 060 | Loss: 0.1312 | Train: 95.73% | Val: 95.57%
Epoch 080 | Loss: 0.1133 | Train: 96.33% | Val: 96.27%
Epoch 099 | Loss: 0.1029 | Train: 96.60% | Val: 96.52%

Training complete.
Best val accuracy : 96.65%  (epoch 97)

Validation confusion matrix
                  Healthy    Infected
  Actual Healthy    590        104        <- FP
  Actual Infected   54         3971       <- FN (critical)
  Precision   : 97.45%
  Recall      : 98.66%
  F1      

In [ ]:
import os

model_dir = "/content/drive/MyDrive/DNA_database_For_training/model_coe"
for f in sorted(os.listdir(model_dir)):
    mb = os.path.getsize(os.path.join(model_dir, f)) / 1_048_576
    status = "OK" if mb > 0 else "EMPTY"
    print(f"  {status}  {f:<45} {mb:.3f} MB")

  OK  best_model.pt                                 0.004 MB
  OK  bias_fp32.npy                                 0.000 MB
  OK  model_fp32.pt                                 0.004 MB
  OK  multiplexdiagnosis_weights_64bit.coe          0.001 MB
  OK  quant_params.txt                              0.000 MB
  OK  training_log.txt                              0.002 MB
  OK  weights_fp32.npy                              0.002 MB
  OK  weights_int8.npy                              0.004 MB


In [ ]:
import os
path = "/content/drive/MyDrive/DNA_database_For_training/model_coe/model_fp32.pt"
print("EXISTS" if os.path.exists(path) else "NOT FOUND")

EXISTS


In [ ]:
"""
encode_and_test_validation_v2.py
=================================
Corrected validation script.

CRITICAL FIX:
  Training used mean k-mer vector of 200 reads per sample.
  This script now does the same — groups reads into samples
  of READS_PER_SAMPLE and averages them before inference.

  Previous version fed individual reads → wrong input distribution
  → 99% false positives even on a good model.

Expected result on pure human 1000 Genomes samples:
  All samples classified as healthy (0).
  Any classified as infected (1) = true false positives.
"""

from google.colab import drive
drive.mount("/content/drive")

import os
import gzip
import numpy as np
import torch
import torch.nn as nn

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR   = "/content/drive/MyDrive/DNA_database_For_training"
VAL_DIR    = f"{BASE_DIR}/validation_human"
MODEL_PATH = f"{BASE_DIR}/model_coe/model_fp32.pt"
#MODEL_PATH = f"{BASE_DIR}/model_v4/model_fp32.pt"
#MODEL_PATH = f"{BASE_DIR}/model_v3/model_fp32.pt"

K                = 4      # must match training
READS_PER_SAMPLE = 50    # must match build_clean_dataset_v3.py
# ─────────────────────────────────────────────────────────────────────────────

VEC_LEN   = 4 ** K
BASE_BITS = {"A": 0, "C": 1, "G": 2, "T": 3}
VALID     = set(BASE_BITS)

SAMPLES = [
    {"file": "NA12878.fastq.gz", "name": "NA12878 (European CEU)"},
    {"file": "NA19238.fastq.gz", "name": "NA19238 (African YRI)"},
    {"file": "HG00096.fastq.gz", "name": "HG00096 (European GBR)"},
]


# ── k-mer encoding ────────────────────────────────────────────────────────────
def kmer_to_index(kmer):
    idx = 0
    for b in kmer:
        idx = (idx << 2) | BASE_BITS[b]
    return idx

def encode_sequence(seq):
    vec = np.zeros(VEC_LEN, dtype=np.float32)
    for i in range(len(seq) - K + 1):
        km = seq[i: i + K]
        if any(b not in VALID for b in km):
            continue
        vec[kmer_to_index(km)] += 1
    # Normalize to frequency — must match build_clean_dataset_v5.py
    total = vec.sum()
    if total > 0:
        vec = vec / total
    return vec

def clean_seq(seq):
    """Same cleaning as build_clean_dataset_v3.py — keep only ACGT."""
    return "".join(b for b in seq.upper() if b in VALID)

def reads_to_sample(reads):
    """
    Average k-mer vectors across all reads in a sample.
    This matches exactly how training samples were built.
    """
    vecs = np.stack([encode_sequence(r) for r in reads], axis=0)
    return vecs.mean(axis=0)


# ── Load sequences from FASTQ.gz ──────────────────────────────────────────────
def load_sequences(path, max_seqs=50_000):
    """Load and clean sequences from a gzipped FASTQ file."""
    seqs    = []
    open_fn = gzip.open if path.endswith(".gz") else open

    with open_fn(path, "rt", errors="replace") as f:
        lines = []
        for line in f:
            lines.append(line.rstrip("\n").rstrip("\r"))
            if len(lines) == 4:
                _, seq, _, _ = lines
                lines = []
                cleaned = clean_seq(seq)
                if len(cleaned) >= K:
                    seqs.append(cleaned)
                if len(seqs) >= max_seqs:
                    break

    return seqs


# ── Build sample vectors from a list of sequences ────────────────────────────
def build_sample_vectors(seqs, reads_per_sample, rng=None):
    """
    Chunk sequences into groups of reads_per_sample.
    Average each group → one sample vector.
    Returns array of shape (n_samples, 256).
    """
    import random
    if rng is None:
        rng = random.Random(42)

    shuffled = seqs[:]
    rng.shuffle(shuffled)

    n_samples = len(shuffled) // reads_per_sample
    if n_samples == 0:
        raise ValueError(
            f"Not enough reads to form one sample. "
            f"Have {len(shuffled)}, need {reads_per_sample}. "
            f"Lower READS_PER_SAMPLE or increase max_seqs."
        )

    X = []
    for i in range(n_samples):
        chunk = shuffled[i * reads_per_sample: (i + 1) * reads_per_sample]
        X.append(reads_to_sample(chunk))

    return np.stack(X, axis=0).astype(np.float32)


# ── Model definition (must match training) ────────────────────────────────────
#class SARSDetector(nn.Module):
 #   def __init__(self, input_dim=256):
 #       super().__init__()
 #       self.layer1 = nn.Linear(input_dim, 128)
 #       self.bn1    = nn.BatchNorm1d(128)
 #       self.layer2 = nn.Linear(128, 64)
 #       self.bn2    = nn.BatchNorm1d(64)
 #       self.layer3 = nn.Linear(64, 2)
 #       self.relu    = nn.ReLU()
 #       self.dropout = nn.Dropout(0.3)

 #   def forward(self, x):
 #       x = self.dropout(self.relu(self.bn1(self.layer1(x))))
 #       x = self.dropout(self.relu(self.bn2(self.layer2(x))))
 #       return self.layer3(x)


class MACArrayModel(nn.Module):
    def __init__(self, input_dim=256, num_classes=2):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(num_classes, input_dim) * 0.01)
        self.bias    = nn.Parameter(torch.zeros(num_classes))

    def forward(self, x):
        return x @ self.weights.T + self.bias

# ── Load model ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(MODEL_PATH, map_location=device)
#model  = SARSDetector(input_dim=ckpt["input_dim"]).to(device)
model  = MACArrayModel(input_dim=ckpt["input_dim"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Device         : {device}")
print(f"Model loaded   : {MODEL_PATH}")
print(f"Trained val acc: {ckpt['best_val_acc']:.2f}%")
print(f"Reads/sample   : {READS_PER_SAMPLE}  (must match training)\n")


# ── Test each sample ──────────────────────────────────────────────────────────
print("=" * 60)
print("  Validation — pure human samples NEVER seen in training")
print("  Input: averaged sample vectors (matches training format)")
print("=" * 60)
print()

total_samples   = 0
total_correct   = 0
total_false_pos = 0

for s in SAMPLES:
    path = os.path.join(VAL_DIR, s["file"])

    if not os.path.exists(path):
        print(f"  SKIP: {s['file']} not found")
        continue

    mb = os.path.getsize(path) / 1_048_576
    if mb < 0.1:
        print(f"  SKIP: {s['file']} is empty ({mb:.2f} MB)")
        continue

    print(f"  Sample : {s['name']}")

    # Load sequences
    seqs = load_sequences(path, max_seqs=50_000)
    print(f"  Sequences loaded : {len(seqs):,}")

    # Build sample vectors — same process as training
    X = build_sample_vectors(seqs, READS_PER_SAMPLE)
    print(f"  Samples built    : {X.shape[0]:,}  "
          f"({READS_PER_SAMPLE} reads averaged per sample)")

    # All are healthy — pure human
    y_true = np.zeros(len(X), dtype=np.int64)

    # Run inference
    with torch.no_grad():
        X_t   = torch.tensor(X).to(device)
        preds = model(X_t).argmax(dim=1).cpu().numpy()

    correct   = int((preds == 0).sum())
    false_pos = int((preds == 1).sum())
    fp_pct    = false_pos / len(preds) * 100

    total_samples   += len(preds)
    total_correct   += correct
    total_false_pos += false_pos

    print(f"  Correctly healthy (0) : {correct:,}")
    print(f"  False positives   (1) : {false_pos:,}  ({fp_pct:.2f}%)")

    if false_pos == 0:
        print(f"  Result : PASS")
    elif fp_pct < 5:
        print(f"  Result : ACCEPTABLE")
    else:
        print(f"  Result : WARN — high false positive rate")
    print()

# ── Overall ───────────────────────────────────────────────────────────────────
if total_samples > 0:
    overall_acc = total_correct / total_samples * 100
    fp_rate     = total_false_pos / total_samples * 100

    print("=" * 60)
    print("  Overall result")
    print("=" * 60)
    print(f"  Total samples tested : {total_samples:,}")
    print(f"  Correct (healthy)    : {total_correct:,}  ({overall_acc:.2f}%)")
    print(f"  False positives      : {total_false_pos:,}  ({fp_rate:.2f}%)")
    print()
    if overall_acc >= 99:
        print("  CONFIRMED — model is genuine.")
        print("  Correctly classifies new unseen human DNA as healthy.")
    elif overall_acc >= 90:
        print("  GOOD — model generalises well to new human samples.")
    else:
        print("  FAIL — still classifying human DNA as infected.")
        print("  The class imbalance (12% healthy vs 88% infected) in")
        print("  ml_ready_v3 is likely the cause.")
        print("  Fix: re-run build_clean_dataset_v3.py with READS_PER_SAMPLE=50")
        print("  to balance the classes before retraining.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device         : cpu
Model loaded   : /content/drive/MyDrive/DNA_database_For_training/model_coe/model_fp32.pt
Trained val acc: 96.65%
Reads/sample   : 50  (must match training)

  Validation — pure human samples NEVER seen in training
  Input: averaged sample vectors (matches training format)

  Sample : NA12878 (European CEU)
  Sequences loaded : 50,000
  Samples built    : 1,000  (50 reads averaged per sample)
  Correctly healthy (0) : 1,000
  False positives   (1) : 0  (0.00%)
  Result : PASS

  Sample : NA19238 (African YRI)
  Sequences loaded : 50,000
  Samples built    : 1,000  (50 reads averaged per sample)
  Correctly healthy (0) : 1,000
  False positives   (1) : 0  (0.00%)
  Result : PASS

  SKIP: HG00096.fastq.gz is empty (0.00 MB)
  Overall result
  Total samples tested : 2,000
  Correct (healthy)    : 2,000  (100.00%)
  False positives      : 0  